# Persistence : 

In [26]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [27]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [28]:
# Define State : 

class JokeState(TypedDict):
    topic : str
    joke  : str 
    explanation : str 

In [29]:
def generate_joke(state : JokeState):
    prompt = f'generate a joke on the topic {state['topic']}'
    response = llm.invoke(prompt).text

    return {
        'joke' : response
    }

In [30]:
def generate_explanation(state : JokeState):
    prompt = f'write an explanation for the joke - {state['joke']}'
    response = llm.invoke(prompt).text

    return {
        'explanaation' : response
    }

In [31]:
# Define graph : 

graph = StateGraph(JokeState)


# add node : 
graph.add_node('generate_joke' , generate_joke)
graph.add_node('generate_explanation' , generate_explanation)

# add edge : 
graph.add_edge(START , 'generate_joke')
graph.add_edge('generate_joke' , 'generate_explanation')
graph.add_edge('generate_explanation' , END)

# CheckPointer : 
checkpointer = InMemorySaver()

# compile : 
workflow = graph.compile(checkpointer=checkpointer)

In [32]:
config1 = {'configurable' : {'thread_id' : "1"}} # when making a checkpointer you have to give unique thread_id per invoke 
workflow.invoke({'topic' : 'wife'},config=config1)  

{'topic': 'wife',
 'joke': 'A husband asks his wife, "If I won the lottery, what would you do?"\n\nShe replies, "I’d take half and leave you!"\n\nThe husband smiles, pulls out a ticket, and says, "Awesome. I won $12. Here’s $6, now get packing."'}

In [33]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'wife', 'joke': 'A husband asks his wife, "If I won the lottery, what would you do?"\n\nShe replies, "I’d take half and leave you!"\n\nThe husband smiles, pulls out a ticket, and says, "Awesome. I won $12. Here’s $6, now get packing."'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1934fe-f2b2-6bdd-8002-952081570ed0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-08T17:38:14.947897+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1934fe-a96e-6f64-8001-3277b7f9491c'}}, tasks=(), interrupts=())

In [34]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'wife', 'joke': 'A husband asks his wife, "If I won the lottery, what would you do?"\n\nShe replies, "I’d take half and leave you!"\n\nThe husband smiles, pulls out a ticket, and says, "Awesome. I won $12. Here’s $6, now get packing."'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1934fe-f2b2-6bdd-8002-952081570ed0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-08T17:38:14.947897+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1934fe-a96e-6f64-8001-3277b7f9491c'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'wife', 'joke': 'A husband asks his wife, "If I won the lottery, what would you do?"\n\nShe replies, "I’d take half and leave you!"\n\nThe husband smiles, pulls out a ticket, and says, "Awesome. I won $12. Here’s $6, now get packing."'}, next=('generate_explanation',), config={'configurable': {'thread_id': 